Parse pcap files to .csv logs

In [ ]:
import os
import dpkt
import gzip
import socket
import base64
import shutil
from datetime import datetime, timedelta
import subprocess
from multiprocessing import Pool
import re
import glob
import polars as pl

In [ ]:
START = datetime(2025,10,1)
def get_day(i,beginning=START):
    DATE = beginning + timedelta(days=i)
    day = DATE.strftime('%d')
    month = DATE.strftime('%m')
    year = DATE.strftime('%Y')
    return day,month,year

In [ ]:
def gen_csv_from_pcap(pcap_file,date):
    try:
        base_name = os.path.basename(pcap_file)
        trace_name = f"trace_{base_name.split('net-')[1].split('.')[0]}.csv"
        os.makedirs(f"./raw_trace/{date}", exist_ok=True)
        out_filename = os.path.join(f"./raw_trace/{date}", trace_name)
        out_unmatched_packets = os.path.join(f"./raw_trace/{date}/unmatched", trace_name)


        if os.path.exists(out_filename):
            return "skipped"

        header = (
            "ts,src_ip,dst_ip,proto,hl,pkt_len,ttl,ipid,"
            "src_port,dst_port,tcp_flags,tcp_seq,tcp_ack,l4_len,tcp_win,tcp_opts,"
            "icmp_type,icmp_code\n"
        )

        with open(out_filename, "w+") as out_file:
            out_file.writelines(header)

        with gzip.open(pcap_file, "rb") as f:
            pcap = dpkt.pcap.Reader(f)
            buffer = []

            for ts, buf in pcap:
                try:
                    eth = dpkt.ethernet.Ethernet(buf)
                    ip = eth.data

                    if not isinstance(ip, dpkt.ip.IP):
                        with open(out_unmatched_packets, "a+") as out_log:
                            out_log.write(f"{ts}, {buf}\n")
                        continue

                    src_ip_bytes = ip.src
                    dst_ip_bytes = ip.dst

                    src_ip = socket.inet_ntoa(src_ip_bytes)
                    dst_ip = socket.inet_ntoa(dst_ip_bytes)

                    # if src_ip in DROP_IPS or dst_ip in DROP_IPS:
                    #     continue

                    # if ip_in_same_subnet(src_ip_bytes, dst_ip_bytes):
                        # continue

                    line = f"{ts},{src_ip},{dst_ip},{ip.p},{ip.hl},{ip.len},{ip.ttl},{ip.id},"
                    if isinstance(ip.data, dpkt.tcp.TCP):
                        tcp = ip.data
                        tcpopts = tcp.opts
                        if len(tcpopts)==0:
                            tcpopts = -1
                        else:
                            tcpopts = base64.b64encode(tcpopts).decode('ascii')
                        line += f"{tcp.sport},{tcp.dport},{tcp.flags},{tcp.seq},{tcp.ack},{tcp.off},{tcp.win},{tcpopts},-1,-1\n"
                    elif isinstance(ip.data, dpkt.udp.UDP):
                        udp = ip.data
                        line += f"{udp.sport},{udp.dport},-1,-1,-1,{udp.ulen},-1,-1,-1,-1\n"
                    elif isinstance(ip.data, dpkt.icmp.ICMP):
                        icmp = ip.data
                        line += f"-1,-1,-1,-1,-1,-1,-1,-1,{icmp.type},{icmp.code}\n"
                    else:
                        line += f"-1,-1,-1,-1,-1,-1,-1,-1,-1,-1\n"

                    buffer.append(line)

                    if len(buffer) >= 10000:
                        with open(out_filename, "a") as out_file:
                            out_file.writelines(buffer)
                        buffer = []

                except Exception as e_inner:
                    print(f"Error parsing a packet: {e_inner}", flush=True)



            if buffer:
                with open(out_filename, "a") as out_file:
                    out_file.writelines(buffer)
        return "success"

    except Exception:
        print(pcap_file,date)
        return "error"

In [ ]:
for D in range(31):
    DD,MM,YY = get_day(D)
    folder_path = f'/path/to/pcap/{YY}/{MM}/{DD}/'
    files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
    success_count = 0
    skipped_count = 0
    error_count = 0
    args = [(file, f'{YY}{MM}{DD}') for file in files]
    with Pool(processes=30) as pool:
        for result in pool.starmap(gen_csv_from_pcap, args):
                if result == "success":
                    success_count += 1
                elif result == "skipped":
                    skipped_count += 1
                elif result == "error":
                    error_count += 1
    print(YY,MM,DD,success_count, skipped_count, error_count)

In [ ]:
def combine_logs(D):
    """Concat logs to 1 day of traffic"""
    DD,MM,YY = get_day(D)
    csv_dir = f'./raw_trace/{YY}{MM}{DD}'
    output_file = f'<save_path>/{YY}{MM}{DD}.csv'
    
    if not os.path.exists(csv_dir):
        print(f"no {csv_dir}")
        return

    # Get list of files and sort by Unix timestamp in filename
    csv_files = glob.glob(f'{csv_dir}/*')
    if not csv_files:
        print(f"no CSV files in {csv_dir}")
        return
    
    # Get list of files and sort by Unix timestamp in filename
    csv_files = glob.glob(f'{csv_dir}/*')
    csv_files.sort(key=lambda f: int(os.path.basename(f).split('-')[1].split('.')[0]))

    # Build shell command: keep header from first file, skip headers in others
    # Build shell command
    cmd_parts = [f'head -n 1 "{csv_files[0]}"']
    cmd_parts += [f'tail -n +2 "{f}"' for f in csv_files]
    cmd = '(' + ' ; '.join(cmd_parts) + f') > "{output_file}"'

    # Run the command
    subprocess.run(cmd, shell=True, check=True)
    print(output_file)

In [ ]:
def compress_csv(D):
    DD,MM,YY = get_day(D)
    file_path = f'./br_traces_daily/br_{YY}{MM}{DD}.csv'

    if not os.path.exists(file_path):
        print(f"no {file_path}\n")
        return
    
    gz_path = file_path + '.gz'
    with open(file_path, 'rb') as f_in, gzip.open(gz_path, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(file_path)

In [ ]:
with Pool(processes=31) as pool:
    pool.map(combine_logs,range(31))

In [ ]:
with Pool(processes=31) as pool:
    pool.map(compress_csv,range(31))